# NUTDTS 816 Time Series Analysis
## L20 Cointegration, VECM, dynamic factor models

Lab notebook for Chapter 10 of the lecture notes. Run the setup cell first. Every code cell reproduces an example from the notes; the exercises at the end are from the chapter's self-check list.

**Instructor:** Dr Tolulope Adesina · NUTM MSc Data Science · 2026

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://raw.githubusercontent.com/<your-github-user>/nutdts816/main"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "nigeria_fx", "nigeria_grid", "nigeria_malaria", "nigeria_rainfall", "bonny_light", "daily_demand",
              "airpassengers", "a10", "h02", "ausbeer", "elecequip", "usmelec", "goog", "nile", "austourists", "oil", "dax", "uschange", "elecdemand"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

### Carried forward from Lab 19 (run these cells first; they define the objects used below)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from statsmodels.tsa.api import VAR
import tsdata
us = tsdata.uschange()[['Consumption', 'Income', 'Production']]
us = us.asfreq('QS')
train, test = us[:'2014-12'], us['2015-01':]          # hold out the last 8 quarters
model = VAR(train)
print(model.select_order(maxlags=8).summary())

In [ ]:
var3 = model.fit(3)
print('Stable (all companion eigenvalues inside unit circle):', var3.is_stable(), '| smallest inverse-root modulus (must exceed 1):', np.round(np.abs(var3.roots).min(), 3))
print('\nEquation for Consumption (growth in consumption depends on lags of all three):')
print(var3.params['Consumption'].round(3).to_string())
w = var3.test_whiteness(nlags=12); print(f'\nResidual Portmanteau test up to lag 12 (H0: no residual autocorrelation): statistic = {w.test_statistic:.1f}, p = {w.pvalue:.3f}')

In [ ]:
h = len(test)
fc = var3.forecast(train.values[-3:], steps=h); fc = pd.DataFrame(fc, index=test.index, columns=us.columns)
lo, mid, hi = var3.forecast_interval(train.values[-3:], steps=h, alpha=0.2)
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for i, col in enumerate(us.columns):
    us[col]['2008':].plot(ax=axes[i], lw=1, label='observed'); fc[col].plot(ax=axes[i], lw=2, color='#B8860B', label='VAR(3) forecast')
    axes[i].fill_between(test.index, lo[:, i], hi[:, i], color='#B8860B', alpha=0.2); axes[i].set_title(col + ' growth (%)'); axes[i].set_xlabel('')
axes[0].legend(fontsize=8)
mae_var = (fc - test).abs().mean(); mae_mean = (train.iloc[-20:].mean() - test).abs().mean()
print(pd.DataFrame({'VAR(3) MAE': mae_var, 'Recent-mean MAE': mae_mean}).round(3).to_string())
_caption = 'Eight-quarter VAR forecasts revert quickly to the long-run means. Growth rates are only modestly forecastable: on this hold-out the VAR ties a recent mean for income and loses narrowly for consumption and production. Quarterly growth rates are close to unforecastable beyond a quarter or two, and a VAR cannot change that.'

In [ ]:
rows = []
for caused in us.columns:
    for causing in us.columns:
        if caused != causing:
            r = var3.test_causality(caused, [causing], kind='f'); rows.append({'does': causing, 'Granger-cause': caused, 'F': round(r.test_statistic, 2), 'p-value': round(r.pvalue, 3)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
irf = var3.irf(10)
fig = irf.plot(orth=True, impulse='Consumption', figsize=(9, 6))
fig.suptitle('Orthogonalised impulse responses to a one-s.d. shock in Consumption growth (ordering: Consumption, Income, Production)', y=1.0)
_caption = 'A positive consumption shock raises income and production growth over the following two or three quarters; the effects die out within a year, as expected for growth rates. The shaded bands are 95% bootstrap intervals.'

In [ ]:
fevd = var3.fevd(8)
print('Share of 8-quarter forecast error variance of each series due to each shock:')
print(pd.DataFrame(fevd.decomp[:, -1, :], index=us.columns, columns=[f'{c} shock' for c in us.columns]).round(3).to_string())

## Cointegration, VECM, dynamic factor models

### 10.5 Cointegration and error correction

In [ ]:
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.tsa.vector_ar.vecm import VECM
rng = np.random.default_rng(10); n = 300
common = np.cumsum(rng.normal(size=n))                                 # a shared random-walk driver
y2 = common + rng.normal(0, 0.5, n)                                    # e.g. the official exchange rate
y1 = 1.3 * common + 2 + rng.normal(0, 0.7, n)                         # e.g. the parallel rate: 1.3 x the driver plus noise
y1 = pd.Series(y1, name='y1'); y2 = pd.Series(y2, name='y2')
print(f'ADF on y1: p = {adfuller(y1)[1]:.3f};  on y2: p = {adfuller(y2)[1]:.3f}  (both I(1))')
t_stat, p_eg, _ = coint(y1, y2)
print(f'Engle-Granger cointegration test: statistic = {t_stat:.2f}, p = {p_eg:.4f}  (H0: no cointegration)')
import statsmodels.api as sm
ols = sm.OLS(y1, sm.add_constant(y2)).fit(); print(f'Estimated long-run relationship: y1 = {ols.params.iloc[0]:.2f} + {ols.params.iloc[1]:.3f} y2   (true slope 1.3)')
vecm = VECM(pd.concat([y1, y2], axis=1), k_ar_diff=1, coint_rank=1, deterministic='co').fit()
print(f'VECM adjustment coefficients alpha: y1 equation = {vecm.alpha[0,0]:.3f}, y2 equation = {vecm.alpha[1,0]:.3f}')
print('When y1 sits above its long-run value, y1 falls (negative alpha) and y2 rises (positive alpha): both movements close the gap, and their sizes say a deviation is largely corrected within two or three periods.')
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(y1.values, lw=1, label='y1'); axes[0].plot(y2.values, lw=1, label='y2'); axes[0].set_title('Two I(1) series that move together'); axes[0].legend(fontsize=8)
axes[1].plot(ols.resid.values, lw=1); axes[1].axhline(0, color='#555555', lw=0.8); axes[1].set_title('Residual of the long-run regression: stationary (the disequilibrium)')
_caption = 'Cointegration: each series is a random walk, their difference (after scaling) is not. The VECM uses the disequilibrium to forecast the correction.'

### 10.6 Dynamic factor models: a conceptual treatment

In [ ]:
from statsmodels.tsa.api import DynamicFactor
us5 = tsdata.uschange(); us5 = us5.asfreq('QS')
z = (us5 - us5.mean()) / us5.std()
dfm = DynamicFactor(z, k_factors=1, factor_order=2).fit(disp=False, maxiter=500)
print('Loadings of each series on the single common factor (standardised data):')
print(pd.Series(dfm.params[[f'loading.f1.{c}' for c in z.columns]].values, index=z.columns).round(3).to_string())
fig, ax = plt.subplots(figsize=(9, 3)); pd.Series(dfm.factors.filtered[0], index=z.index).plot(ax=ax, lw=1.2); ax.axhline(0, color='#555555', lw=0.8)
ax.set_title('Estimated common factor for the five US series: a business-cycle indicator'); ax.set_xlabel('')
_caption = 'A one-factor DFM on five quarterly series. Consumption, income and production load positively, unemployment negatively: the factor is the business cycle, with recessions as the troughs.'

## Exercises

4. Simulate two independent random walks and run the Engle-Granger test. Then simulate a cointegrated pair with adjustment coefficient $\alpha_1 = -0.3$ and run it again. Report both outcomes and explain what $\alpha_1 = -0.3$ means for how long a deviation persists.
5. Describe the Cholesky ordering you would use for a VAR in oil prices, the exchange rate and inflation, and defend it.
6. Explain in one paragraph how a DFM makes nowcasting possible when GDP is quarterly and the indicators are monthly.

In [ ]:
# Your work here
